# **Lab 1 – Applied Statistics for Data Science**

***Author:*** Leandro Ariza (larizaj@eafit.edu.co)
***Students*** Jeronimo Velasquez Escobar jvelasquee@eafit.edu.co
***Students*** Manuela Caro Villada mcarov@eafit.edu.co



## 0. General Instructions

- Work individually or in groups of up to 2 people.
- Submit a Jupyter or Colab notebook, clearly organized and commented.
- Interpret your results thoughtfully — code alone is not enough.
- Datasets will be provided, or students may simulate synthetic data when required.
- You may use libraries like pandas, numpy, matplotlib, seaborn, scipy — but do not use high-level statistical modeling libraries for tasks you're expected to implement.
- File naming convention: `AE-Lab_1-FirstName_LastName_1-FirstNAme_LastName_2.ipynb`.
- **Submission deadline: March 15, 2026 at midnight.**

## 1. Building a Naive Bayes Spam Filter (from scratch)

### 1.1. Introduction

In this exercise, you will implement a simple **Naive Bayes classifier** from scratch to detect whether an email is **spam** or **not spam**.

Each email will be represented using **binary features** that indicate whether certain keywords appear in the message:

* `1` = the keyword is present
* `0` = the keyword is absent

To build the classifier, we will assume that features are **conditionally independent given the class**. Under this assumption, you will estimate:

* the **prior probabilities** of each class,
* the **conditional probabilities** of each feature given the class,
* and the **posterior scores** used to classify a new email.

At the end, you will use your implementation to classify unseen emails and reflect on the assumptions and limitations of this approach.

### 1.2. On the Naive Bayes Classifier

#### 1.2.1. The big idea

Naive Bayes is a **probabilistic classifier** based on **Bayes’ theorem**. Its goal is to answer the following question:

> Given a set of observed features, how likely is each possible class?

In this homework, the question becomes:

> Given the keywords present in an email, how likely is it that the email is spam?

This same idea appears in many applications. For example:

* Is an email **spam** or **not spam**?
* Will a customer **churn** or **stay**?
* Is a review **positive** or **negative**?

#### 1.2.2. Bayes’ theorem for classification

**Definition**

Let $C$ denote the (discrete) class variable and let $X_1, X_2, \ldots, X_n$ denote the features.

Bayes’ theorem states that

>$$P(C = c_i \mid X_1, X_2, \ldots, X_n) = \frac{P(X_1, X_2, \ldots, X_n \mid C = c_i),P(C = c_i)}{P(X_1, X_2, \ldots, X_n)}$$

This expression tells us how to update our belief about the class after observing the features.

**Interpretation of each term**

* **Posterior**
  $$
  P(C = c_i \mid X_1, X_2, \ldots, X_n)
  $$
  This is the probability that the observation belongs to class (c_i) after seeing the features. It is the quantity we want to compare across classes.

* **Prior**
  $$
  P(C = c_i)
  $$
  This measures how common class (c_i) is before looking at any features.

* **Likelihood**
  $$
  P(X_1, X_2, \ldots, X_n \mid C = c_i)
  $$
  This is the probability of observing the feature pattern assuming the class is already known.

* **Evidence**
  $$
  P(X_1, X_2, \ldots, X_n)
  $$
  This is a normalizing constant that ensures the posterior probabilities sum to 1 across classes.

#### 1.2.3. Why the model is called *naive*

The difficult part in Bayes’ theorem is the likelihood term

$$
P(X_1, X_2, \ldots, X_n \mid C = c_i)
$$

because modeling the full joint distribution of many features can be very hard.

Naive Bayes makes a simplifying assumption:

> Once the class is known, the features are conditionally independent of one another.

Under this assumption,

$$
P(X_1, X_2, \ldots, X_n \mid C = c_i) = P(X_1 \mid C = c_1) \cdot P(X_2 \mid C = c_i) \cdots P(X_n \mid C = c_i)
$$

which can be written more compactly as

$$
P(X_1, X_2, \ldots, X_n \mid C = c_i) = \prod_{j=1}^n P(X_j \mid C = c_i)
$$

Substituting this into Bayes’ theorem gives

>$$
>P(X_1, X_2, \ldots, X_n \mid C = c_i) =
>\frac{P(C = c_i) \cdot \prod_{j=1}^{n} P(X_j \mid C = c_i)}{P(X_1, X_2, \ldots, X_n)}
>$$

#### 1.2.4. A Scoring-based pproach

In Bayes’ theorem, the denominator $P(X_1, X_2, \ldots, X_n)$ is the same for all classes because it does not depend on the class itself.

This means that, instead of computing the exact posterior probability, we can work with a posterior score for each class:

>$$
>\text{score}(X_1, X_2, \ldots, X_n \mid C = c_i) =  P(C = c_i) \cdot \prod_{j=1}^{n} P(X_j \mid C = c_i)
>$$

This score is proportional to the true posterior probability $P(C = c_i \mid X_1, X_2, \ldots, X_n)$.

Finally, to make predictions given a set of features (for example, deciding whether an email is spam based on the existence of keywords), we compute the score for each class and assign the observation to the class with the largest score.

#### 1.2.5. The Bernoulli setting used in this implementation

In this exercise, each feature is binary: a keyword is either present or absent. This corresponds to a **Bernoulli Naive Bayes** model.

For a feature $X_j$ taking values from $\{0, 1\}$, we estimate:

$$
P(X_j = 1 \mid C = c_i)
$$

from the training data.

Then,

$$
P(X_j = 0 \mid C = c_i) = 1 - P(X_j = 1 \mid C = c_i)
$$

So when evaluating a new email, each feature contributes:

* $P(X_j = 1 \mid C = c_i)$ if the keyword is present
* $P(X_j = 0 \mid C = c_i)$ if the keyword is absent

Using these probabilities, we can compute a score for each class using all observed features.

#### 1.2.6. Why It Works Well

* **Simple and fast**: Training only requires counting frequencies.
* **Works surprisingly well**: Even when the independence assumption is violated, the classifier often performs competitively.
* **Robust to irrelevant features**: Noisy features tend to affect all classes similarly, so their impact often cancels out.

## 2. Toy Dataset

Below is the training dataset we will use to build our Naive Bayes spam filter.

Each row represents an **email**, and each column is a **binary feature** indicating whether a specific keyword appears in the email:

* `1` → the keyword appears in the email
* `0` → the keyword does not appear

The column **`spam`** indicates whether the email is spam (`1`) or not (`0`).

**Load the dataset**

In [129]:
import pandas as pd

data = [
[1,1,1,0,0,0,1],
[0,1,1,1,0,0,1],
[1,0,1,1,0,0,1],
[0,0,1,0,1,0,1],
[1,1,0,1,0,0,1],
[0,0,0,0,1,1,0],
[0,0,0,0,0,1,0],
[0,1,0,0,1,0,0],
[0,0,0,0,1,0,0],
[0,0,0,1,0,1,0],
[1,0,0,0,1,0,0],
[0,1,0,1,0,0,1],
[1,0,0,0,0,0,1],
[0,0,1,0,0,1,0],
]

columns = [
"has_winner",
"has_money",
"has_click",
"has_offer",
"has_hello",
"has_meeting",
"spam"
]

df = pd.DataFrame(data, columns=columns)
df

,has_winner,has_money,has_click,has_offer,has_hello,has_meeting,spam
0,1,1,1,0,0,0,1
1,0,1,1,1,0,0,1
2,1,0,1,1,0,0,1
3,0,0,1,0,1,0,1
4,1,1,0,1,0,0,1
5,0,0,0,0,1,1,0
6,0,0,0,0,0,1,0
7,0,1,0,0,1,0,0
8,0,0,0,0,1,0,0
9,0,0,0,1,0,1,0


## 3. Step-by-step Implementation

### Step 1: Compute Prior Probabilities

The **prior probability** of a class is the proportion of training examples that belong to that class.

In this dataset, compute:

* $P(\text{spam} = 1)$
* $P(\text{spam} = 0)$

Recall that priors are estimated from the training data as relative frequencies.

In [130]:
# Your code here:
prior_spam_1 = df.value_counts("spam")[1] / len(df)
prior_spam_0 = df.value_counts("spam")[0] / len(df)

print("P(spam=1):", prior_spam_1)
print("P(spam=0):", prior_spam_0)

P(spam=1): 0.5
P(spam=0): 0.5


> **Reflection:**  
>- Do the priors suggest that spam and non-spam emails are balanced in this dataset?
>- How might class imbalance affect the predictions of a Naive Bayes classifier?

The answer to the firts question it's yes,  they are balanced. As we see in before, the prior probabilities in for each class it's 50% for spam and 50% for no spam. If the dataset were imbalanced, the Naive Bayes classifier could become biased toward the majority class. This happens because the prior probability (P(C=c_i)) would be larger for the class that appears more frequently in the training data. As a result, the model might predict the majority class more often, even when the evidence from the features suggests otherwise. This could lead to poor performance when identifying the minority class, since the classifier would tend to favor the class with the higher prior probability.


### Step 2: Estimate conditional probabilities

For Naive Bayes, we need to estimate probabilities of the form

$$
P(X_j = 1 \mid \text{spam}=1)
$$

and

$$
P(X_j = 1 \mid \text{spam}=0)
$$

for each feature $X_j$.

These probabilities represent the likelihood that a given keyword appears in an email conditional on the class of the email.

In practice, we estimate these probabilities using relative frequencies within each class in the training dataset.

**Start with one feature**

As a first example, compute:

$$
P(\text{has\_winner} = 1 \mid \text{spam}=1)
$$

This is the proportion of spam emails that contain the word "winner".

In [131]:
# Your code here
p_winner_given_spam = df[df['spam']==1]['has_winner'].mean()
p_winner_given_spam

np.float64(0.5714285714285714)

Now compute

$$
P(\text{has\_winner} = 1 \mid \text{spam}=0)
$$

This represents the proportion of non-spam emails that contain the word "winner".

In [132]:
# Your code here
p_winner_given_not_spam = df[df['spam']==0]['has_winner'].mean()
p_winner_given_not_spam

np.float64(0.14285714285714285)

> **Reflection**
>
>- Is the word winner more common in spam or non-spam emails?
>- Based on this probability alone, would the presence of winner increase or decrease the likelihood that an email is spam?

The word winner is more common in spam emails. Based on this, the word winner increases the likelihood that an email is spam.

**Avoiding zero probabilities: Laplace smoothing**

In small datasets, it is possible that a feature **never appears with a certain class**. In that case, the estimated probability would become **exactly zero**.

Since Naive Bayes multiplies many probabilities together, a single zero would make the entire class score equal to zero.

To avoid this issue, we use **Laplace smoothing**, which slightly adjusts the probability estimates.

Instead of computing the probability as a simple frequency, we use

$$
P(X_j = 1 \mid C=c) =
\frac{\text{count}(X_j = 1, C=c) + 1}
{\text{count}(C=c) + 2}
$$

The denominator uses **+2** because the feature can take **two possible values** (0 or 1).

Modify your implementation above to incorporate this adjustment.

In [133]:
# Your code here
spam = df[df["spam"] == 1]

prob = (spam["has_winner"].sum() + 1) / (len(spam) + 2)
p_winner_given_spam = prob

not_spam = df[df["spam"] == 0]

prob = (not_spam["has_winner"].sum() + 1) / (len(not_spam) + 2)
p_winner_given_not_spam = prob
print("P(has_winner=1 | spam=1):", p_winner_given_spam)
print("P(has_winner=1 | spam=0):", p_winner_given_not_spam)

P(has_winner=1 | spam=1): 0.5555555555555556
P(has_winner=1 | spam=0): 0.2222222222222222


**Generalize to all features**

Now repeat the same calculation for **all features** in the dataset.

Instead of writing the code manually for each keyword, you can use a **loop** to iterate through the feature columns.

Recall that the features in the dataset are:

* `has_winner`
* `has_money`
* `has_click`
* `has_offer`
* `has_hello`
* `has_meeting`

For each feature $X_j$, compute the smoothed probabilities:

$$
P(X_j = 1 \mid \text{spam}=1)
$$

and

$$
P(X_j = 1 \mid \text{spam}=0)
$$

using **Laplace smoothing**.

A convenient way to store the results is in a **dictionary**, where:

* the **key** is the feature name
* the **value** contains the two conditional probabilities

For example:

```python
feature_probs = {
    "has_winner": {
        "spam": ...,
        "not_spam": ...
    },
    ...
}
```

You may find it useful to:

* iterate over the feature columns in the dataframe
* compute the counts within each class
* apply the Laplace smoothing formula introduced earlier

In [134]:
feature_probs = {}

spam_df = df[df["spam"] == 1]
not_spam_df = df[df["spam"] == 0]

for feature in columns[:-1]:

    p_feature_given_spam = (spam_df[feature].sum() + 1) / (len(spam_df) + 2)
    p_feature_given_not_spam = (not_spam_df[feature].sum() + 1) / (len(not_spam_df) + 2)

    feature_probs[feature] = {
        "spam": p_feature_given_spam,
        "not_spam": p_feature_given_not_spam
    }

for feature, probs in feature_probs.items():
    print(feature)
    print(f"Probabilidades {probs}")
    print(probs['spam'] - probs['not_spam'])

has_winner
Probabilidades {'spam': np.float64(0.5555555555555556), 'not_spam': np.float64(0.2222222222222222)}
0.33333333333333337
has_money
Probabilidades {'spam': np.float64(0.5555555555555556), 'not_spam': np.float64(0.2222222222222222)}
0.33333333333333337
has_click
Probabilidades {'spam': np.float64(0.5555555555555556), 'not_spam': np.float64(0.2222222222222222)}
0.33333333333333337
has_offer
Probabilidades {'spam': np.float64(0.5555555555555556), 'not_spam': np.float64(0.2222222222222222)}
0.33333333333333337
has_hello
Probabilidades {'spam': np.float64(0.2222222222222222), 'not_spam': np.float64(0.5555555555555556)}
-0.33333333333333337
has_meeting
Probabilidades {'spam': np.float64(0.1111111111111111), 'not_spam': np.float64(0.5555555555555556)}
-0.4444444444444445


> **Reflection:**  
> - Which features have the largest difference between $P(X_j=1 \mid \text{spam}=1)$ and $P(X_j=1 \mid \text{spam}=0)$?  
> - Why are those useful for classification?

Based on the results from the subtraction, we can say that the "has_meeting" feature is the one with the biggest difference between $P(X_j=1 \mid \text{spam}=1)$ and $P(X_j=1 \mid \text{spam}=0)$. 

It's useful for classification because they help distinguish between spam and non-spam emails. In this case, the feature has_meeting shows the largest difference, meaning that the presence of the word "meeting" is much more associated with one class than the other

### Step 3: Compute Posterior Scores for a New Email

Suppose we receive a new email with the following feature values:

| has_winner | has_money | has_click | has_offer | has_hello | has_meeting |
| ---------- | --------- | --------- | --------- | --------- | ----------- |
| 1          | 1         | 0         | 0         | 1         | 0           |

We now want to compute a **score for each class** using the Naive Bayes model.

Recall that the score for a class $c_i$ is

$$
\text{score}(X_1, X_2, \ldots, X_n \mid C = c_i) = P(C=c)\prod_{j=1}^{n} P(X_j \mid C=c_i)
$$

Because our features are **binary**, we must use:

* $P(X_j = 1 \mid C=c_i)$ if the feature is **present**
* $P(X_j = 0 \mid C=c_i) = 1 - P(X_j = 1 \mid C=c_i)$ if the feature is **absent**

Therefore, the score for each class is obtained by multiplying:

* the **prior probability of the class**
* the **appropriate conditional probability for each feature**

**Define the new email**

In [135]:
new_email = {
    "has_winner": 1,
    "has_money": 1,
    "has_click": 0,
    "has_offer": 0,
    "has_hello": 1,
    "has_meeting": 0
}

**Compute the class scores**

Use the previously computed priors and conditional probabilities to calculate:

In [136]:
# Your code here
score_spam = prior_spam_1
score_not_spam = prior_spam_0
spam_contributions = {}
not_spam_contributions = {}

for feature, value in new_email.items():
    p_spam = feature_probs[feature]['spam']
    p_not_spam = feature_probs[feature]['not_spam']

    if value == 1:
        contrib_spam = p_spam
        contrib_not_spam = p_not_spam
    else:
        contrib_spam = (1 - p_spam)
        contrib_not_spam = (1 - p_not_spam)

    spam_contributions[feature] = contrib_spam
    not_spam_contributions[feature] = contrib_not_spam

    score_spam *= contrib_spam
    score_not_spam *= contrib_not_spam
        
    
print(f"Spam Score: {score_spam}")
print(f"Not Spam Score: {score_not_spam}")
print(f"The feature who contributed the most to the spam score was: {max(spam_contributions, key=spam_contributions.get)}")
print(f"The feature who contributed the most to the not spam score was: {max(not_spam_contributions, key=not_spam_contributions.get)}")


Spam Score: 0.006021364554108546
Not Spam Score: 0.003688085789391484
The feature who contributed the most to the spam score was: has_meeting
The feature who contributed the most to the not spam score was: has_click


*Hint:*

* Use a **multiplication accumulator** to build the score.
* Start each score with the **class prior**.
* Loop over the features of the email.
* For each feature:
  * if the value is **1**, multiply by $P(X_j=1 \mid C=c_i)$
  * if the value is **0**, multiply by $1 - P(X_j=1 \mid C=c_i)$

  where $C=c_i$ denotes the class currently being scored, that is, either spam (`spam = 1`) or not spam (`spam = 0`).

**Make the prediction**

The predicted class corresponds to the **largest score**.

In [137]:
if score_not_spam < score_spam:
    prediction = "It's spam"
else:
    prediction = "It is not spam"
print(f"The email {prediction}")

The email It's spam


In [138]:
new_email['has_money'] = 0
# Your code here
score_spam = prior_spam_1
score_not_spam = prior_spam_0
spam_contributions = {}
not_spam_contributions = {}

for feature, value in new_email.items():
    p_spam = feature_probs[feature]['spam']
    p_not_spam = feature_probs[feature]['not_spam']

    if value == 1:
        contrib_spam = p_spam
        contrib_not_spam = p_not_spam
    else:
        contrib_spam = (1 - p_spam)
        contrib_not_spam = (1 - p_not_spam)

    spam_contributions[feature] = contrib_spam
    not_spam_contributions[feature] = contrib_not_spam

    score_spam *= contrib_spam
    score_not_spam *= contrib_not_spam
        
    
print(f"Spam Score: {score_spam}")
print(f"Not Spam Score: {score_not_spam}")
print(f"The feature who contributed the most to the spam score was: {max(spam_contributions, key=spam_contributions.get)}")
print(f"The feature who contributed the most to the not spam score was: {max(not_spam_contributions, key=not_spam_contributions.get)}")

if score_not_spam < score_spam:
    prediction = "It's spam"
else:
    prediction = "It is not spam"
print(f"The email {prediction}")



Spam Score: 0.004817091643286835
Not Spam Score: 0.012908300262870197
The feature who contributed the most to the spam score was: has_meeting
The feature who contributed the most to the not spam score was: has_money
The email It is not spam


> **Reflection**
> 
> * Does the classifier predict this email as **spam or not spam**?
> * Which features contributed most strongly to the final score?
> * How would the prediction change if the email did not contain the word **money**?

The classifier predict the email as spam, the features tha contributed the most to the score were has_meeting for the spam score and for the not spam score was has_click.

To answer the last question i did this:

```python 

new_email['has_money'] = 0
# Your code here
score_spam = prior_spam_1
#Rest of the code...

```

This is what i already did in the begining but changing the value of the feature "has_money" to 0. And this was the results:
```
Spam Score: 0.004817091643286835
Not Spam Score: 0.012908300262870197
The feature who contributed the most to the spam score was: has_meeting
The feature who contributed the most to the not spam score was: has_money
The email It is not spam
```
The changes that we can notice are that the most important feature for the not_spam score change from "has_click" to "has_money". Also the most important change is now the email is classified as "not spam".


### Step 4: Create a Reusable Prediction Function

So far, we computed the score for **one email manually**.

To make our classifier more useful, we will now implement a **prediction function** that takes an email as input and returns the predicted class.

Your function should:

1. Start with the class priors.
2. Loop over the features in the email.
3. Multiply by the appropriate conditional probabilities.
4. Return the predicted class.

Function template:

In [139]:
def predict_naive_bayes(email : dict, priors : dict, feature_probs : dict):
    spam_contributions = {}
    not_spam_contributions = {}
    score_spam = priors["spam"]
    score_not_spam = priors["not_spam"]
    
    for feature, value in email.items():
        p_spam = feature_probs[feature]['spam']
        p_not_spam = feature_probs[feature]['not_spam']

        if value == 1:
            score_spam *= p_spam
            score_not_spam *= p_not_spam
            spam_contributions[feature] = p_spam # This contribution variables is to save what feature influence the most the predictions
            not_spam_contributions[feature] = p_not_spam
        else:
            spam_contributions[feature] = (1 - p_spam)
            not_spam_contributions[feature] = (1 - p_not_spam)
            score_spam *= (1 - p_spam)
            score_not_spam *= (1 - p_not_spam)

    ratio = score_spam / score_not_spam # This is a way to know how sure the model is with the prediction, if the ratio is big we know that is spam, 
    # if the ratio is small we know that is not spam. But if is close to one we are unsure.
    max_score_spam_contributions = max(spam_contributions, key=spam_contributions.get)
    max_score_not_spam_contributions = max(not_spam_contributions, key=not_spam_contributions.get)

    if score_not_spam < score_spam:

        prediction = f'The email is spam, with a ratio of {ratio} and the feature that contributed the most is {max(spam_contributions, key=spam_contributions.get)}\n the spam score was {score_spam}  and the not spam score was {score_not_spam}'
    else:
        prediction = f'The email is not spam, with a ratio of {ratio} and the feature that contributed the most is {max(not_spam_contributions, key=not_spam_contributions.get)}\n the spam score was {score_spam}  and the not spam score was {score_not_spam}'
    
    return prediction


Example usage:

In [140]:
# Your code here
priors = {
    "spam": 0.5,
    "not_spam": 0.5
}

prediction = predict_naive_bayes(new_email, priors, feature_probs)
print(prediction)# Is not spam because we change the new_email['has_money'] to 0

The email is not spam, with a ratio of 0.3731778425655975 and the feature that contributed the most is has_money
 the spam score was 0.004817091643286835  and the not spam score was 0.012908300262870197


### Step 5: Batch Testing

Now test your classifier on multiple new emails:

| has_winner | has_money | has_click | has_offer | has_hello | has_meeting |
| ---------- | --------- | --------- | --------- | --------- | ----------- |
| 1          | 1         | 1         | 0         | 0         | 0           |
| 0          | 0         | 0         | 0         | 1         | 1           |
| 1          | 0         | 0         | 0         | 1         | 0           |

Represent them in Python and use your prediction function to classify each email.

In [141]:
# Your code here
emails = [
    {
        'has_winner': 1,
        'has_money' : 1,
        'has_click' : 1,
        'has_offer' : 0,
        'has_hello' : 0,
        'has_meeting' : 0
    },
    {
        'has_winner': 0,
        'has_money' : 0,
        'has_click' : 0,
        'has_offer' : 0,
        'has_hello' : 1,
        'has_meeting' : 1
    }
    ,
    {
        'has_winner': 1,
        'has_money' : 0,
        'has_click' : 0,
        'has_offer' : 0,
        'has_hello' : 1,
        'has_meeting' : 0
    }
]
priors = {
    "spam": 0.5,
    "not_spam": 0.5
}
for email in emails:
    prediction = predict_naive_bayes(email, priors, feature_probs)
    print(prediction)

The email is spam, with a ratio of 31.250000000000007 and the feature that contributed the most is has_meeting
 the spam score was 0.026343469924224892  and the not spam score was 0.0008429910375751964
The email is not spam, with a ratio of 0.008529779258642229 and the feature that contributed the most is has_winner
 the spam score was 0.00048170916432868356  and the not spam score was 0.05647381365005712
The email is not spam, with a ratio of 0.3731778425655975 and the feature that contributed the most is has_money
 the spam score was 0.004817091643286835  and the not spam score was 0.012908300262870197


>**Reflection**
>
>- Which email seems most clearly spam?
>- Which email seems most clearly non-spam?
>- Which email contains conflicting evidence and is therefore harder to classify?
>- Which features appear to influence the predictions the most?

Based on the computed scores and their ratios, we can analyze how clearly each email belongs to a class.

The first email is the most clearly **spam**, since it has the largest ratio between the spam and not-spam scores (31.25). This means the spam score is much larger than the not-spam score, indicating strong evidence for the spam class.

The second email is the most clearly **non-spam**, because its ratio is very small (0.0085). In this case, the not-spam score is much larger than the spam score, which strongly supports the non-spam classification.

The third email is the **hardest to classify**, since its ratio (0.373) is the closest to 1. This indicates that the spam and not-spam scores are relatively similar, meaning the model receives more conflicting evidence from the features.

Regarding feature influence, the features that contributed the most to each prediction were:

* **has_meeting** for the first email,
* **has_winner** for the second email,
* **has_money** for the third email.

These features had the largest contribution to the respective class scores, which means they provided the strongest evidence during the classification process.




## 4. Reflection and Extension

In this lab, you implemented a **Naive Bayes classifier from scratch** and used it to detect spam emails based on the presence of certain keywords.

Although the model is simple, it illustrates several important ideas from this unit, including **events, conditional probability, random variables, and probabilistic modeling**.

### 4.1. Reflection: Understanding the Model

Answer the following questions based on the spam dataset used in the lab.

**1. Feature Independence**

Naive Bayes assumes that all features are **conditionally independent given the class**.

* Do you think this assumption is realistic for email text?
* Can you think of examples where two keywords might be strongly related?

Naive Bayes assumes that all features are conditionally independent given the class. In practice, this assumption is not entirely realistic for email text, since many words tend to appear together and are therefore correlated.

For example, keywords such as money, offer, and winner may frequently appear together in spam emails. Similarly, words like hello and meeting may appear together in normal work-related emails. Because of these relationships, the independence assumption is often violated in real text data.

**2. Model Interpretability**

One advantage of Naive Bayes is that it is **easy to interpret**.

* Which features appeared to be the strongest indicators of spam?
* Which features seemed to suggest non-spam emails?

One advantage of Naive Bayes is that it allows us to easily see which features influence the predictions.

In this dataset, features such as has_winner, has_money, and has_offer appeared to be strong indicators of spam, since their probabilities were higher for the spam class. On the other hand, features such as has_hello and has_meeting seemed to suggest non-spam emails, since they were more common in the non-spam class.

This makes the model easy to interpret because we can directly observe which words increase or decrease the likelihood of an email being classified as spam.

**3. Effect of Small Datasets**

In this exercise we used **Laplace smoothing** to adjust the probability estimates.

* Why was this necessary?
* What problem would occur without smoothing?

Laplace smoothing was necessary because the dataset is very small. In small datasets, it is possible that a feature never appears with a certain class, which would cause the estimated probability to be zero.

Without smoothing, a single zero probability would make the entire Naive Bayes score equal to zero because the model multiplies probabilities across features. Laplace smoothing prevents this issue by slightly adjusting the probability estimates so that no probability becomes exactly zero.

**4. Limitations of the Model**

Suppose you wanted to build a **real spam detection system**.

* What limitations does this toy model have?
* What additional features or data would improve the classifier?

This toy model has several limitations compared to a real spam detection system. First, it uses a very small dataset, which makes the probability estimates unreliable. Second, it only considers a few binary features, while real emails contain thousands of possible words and patterns. Additionally, the model assumes feature independence, which is often not true for natural language.

To improve the classifier, we could use a much larger dataset and include more informative features such as the frequency of words, email metadata, sender information, links, attachments, and text structure. Using more advanced models and richer representations of the email content would also improve performance.


### 4.2. Applied Extension — Build Your Own Dataset

To reinforce the ideas learned in this lab, create a **small binary classification dataset of your own** and apply the Naive Bayes workflow you implemented.

**Dataset Requirements**

Design a small binary classification problem and construct a dataset with the following structure:

- Each row represents an observation
- Each column represents a binary feature
- The final column represents the class label

For example, you might create a dataset to classify:

- Movie/videogames reviews (positive vs negative)
- Messages (urgent vs non‑urgent)
- Products (likely to sell vs unlikely to sell)
- Clients (will churn vs will stay)
- Medical records (disease present vs disease absent)
- Customer feedback (satisfied vs unsatisfied)
- Images (cat vs dog)
- Transactions (fraudulent vs legitimate)
- Job applications (qualified vs not qualified)
- Sensor readings (normal vs anomalous)

Your dataset must contain:

* at least **10 observations**
* at least **4 binary features**
* one **binary target variable**

Additionally:

* Your dataset must include at least one feature that appears in both classes. In other words, **no feature should perfectly determine the class label**.
* Include at least **one test examples with conflicting evidence** (some features suggesting each class)

**Tasks**

Using your dataset:

1. Compute the **prior probabilities**.
2. Estimate **conditional probabilities** using Laplace smoothing.
3. Use your **prediction function** to test your classifier on new examples.
4. Design a set of test example where the classifier receives conflicting evidence (some features suggest class 1 while others suggest class 0).


**Reflection**

Answer the following questions:

* What do the prior probabilities reveal about your dataset?
* Which features were **most informative** in your dataset?
* Did you observe any **zero-probability issues** before smoothing?
* Did Laplace smoothing affect any probability estimates?
* Do you think the **independence assumption** holds for the features you designed? Can you identify two features that might be correlated?
* Regarding the ambiguous test examples, what prediction does the model make?

In [142]:
# 4.2 How to identify an spy
import pandas as pd

data = [
# travel lang encrypt combat contacts fake_id weapon no_social tech spy
[1,1,1,1,1,1,1,1,0,1,1],
[1,1,1,1,1,1,0,1,1,1,1],
[1,1,1,0,1,1,1,1,0,0,1],
[1,0,1,1,1,1,1,1,1,1,1],
[1,1,1,1,0,1,1,0,0,1,1],
[1,1,1,0,1,1,0,1,1,1,1],
[1,0,1,1,1,0,1,1,0,1,1],
[1,1,0,1,1,1,1,0,0,1,1],
[0,0,0,0,0,0,0,0,1,0,0],
[0,1,0,0,0,0,0,0,1,0,0],
[0,0,0,0,1,0,0,0,0,0,0],
[0,0,0,1,0,0,0,1,1,0,0],
[0,1,0,0,1,0,0,0,1,0,0],
[1,0,0,0,0,0,0,0,1,1,0],
[0,0,1,0,0,0,0,0,1,0,0],
[0,1,0,0,0,0,1,0,1,0,0],
[0,0,0,0,0,0,0,1,0,0,0],
[0,1,0,1,0,0,0,0,1,0,0],
[1,0,0,0,1,0,0,0,1,1,0],
[0,0,0,0,0,0,0,0,0,1,0]
]

columns = [
"travels_often",
"speaks_many_languages",
"uses_encryption",
"combat_training",
"suspicious_contacts",
"fake_identity",
"carries_weapon",
"avoids_social_media",
"works_in_tech",
"spy_in_family",
"spy"
]

df = pd.DataFrame(data, columns=columns)

df

,travels_often,speaks_many_languages,uses_encryption,combat_training,suspicious_contacts,fake_identity,carries_weapon,avoids_social_media,works_in_tech,spy_in_family,spy
0,1,1,1,1,1,1,1,1,0,1,1
1,1,1,1,1,1,1,0,1,1,1,1
2,1,1,1,0,1,1,1,1,0,0,1
3,1,0,1,1,1,1,1,1,1,1,1
4,1,1,1,1,0,1,1,0,0,1,1
5,1,1,1,0,1,1,0,1,1,1,1
6,1,0,1,1,1,0,1,1,0,1,1
7,1,1,0,1,1,1,1,0,0,1,1
8,0,0,0,0,0,0,0,0,1,0,0
9,0,1,0,0,0,0,0,0,1,0,0


In [143]:
#Takes the priors
prior_spy_1 = df.value_counts("spy")[1] / len(df)
prior_spy_0 = df.value_counts("spy")[0] / len(df)
print(prior_spy_0)
print(prior_spy_1)

0.6
0.4


In [152]:
# Searching zero-probability issues before smoothing
feature_problems = []
probs_no_smooting = {}
for column in columns[:-1]:
    spy = df[df['spy']==1][column].mean()
    not_spy = df[df['spy']==0][column].mean()
    if not_spy == 0:
        feature_problems.append(column)
    if spy == 0:
        feature_problems.append(column)
    x = {'spy': spy, 'not_spy': not_spy}
    probs_no_smooting[column] = x
    

print(f"Columns with problem{feature_problems}")
print("\n")
for feature, probs in probs_no_smooting.items():
    print(f"For the feature {feature}")
    print(f"Probs :{ probs}\n")


Columns with problem['fake_identity']


For the feature travels_often
Probs :{'spy': np.float64(1.0), 'not_spy': np.float64(0.16666666666666666)}

For the feature speaks_many_languages
Probs :{'spy': np.float64(0.75), 'not_spy': np.float64(0.3333333333333333)}

For the feature uses_encryption
Probs :{'spy': np.float64(0.875), 'not_spy': np.float64(0.08333333333333333)}

For the feature combat_training
Probs :{'spy': np.float64(0.75), 'not_spy': np.float64(0.16666666666666666)}

For the feature suspicious_contacts
Probs :{'spy': np.float64(0.875), 'not_spy': np.float64(0.25)}

For the feature fake_identity
Probs :{'spy': np.float64(0.875), 'not_spy': np.float64(0.0)}

For the feature carries_weapon
Probs :{'spy': np.float64(0.75), 'not_spy': np.float64(0.08333333333333333)}

For the feature avoids_social_media
Probs :{'spy': np.float64(0.75), 'not_spy': np.float64(0.16666666666666666)}

For the feature works_in_tech
Probs :{'spy': np.float64(0.375), 'not_spy': np.float64(0.75)}

For the

In [145]:
# Takes the probs
spy_probs = {}

spy_df = df[df["spy"] == 1]
not_spy_df = df[df["spy"] == 0]

for feature in columns[:-1]:

    p_feature_given_spy = (spy_df[feature].sum() + 1) / (len(spy_df) + 2)
    p_feature_given_not_spy = (not_spy_df[feature].sum() + 1) / (len(not_spy_df) + 2)

    spy_probs[feature] = {
        "spy": p_feature_given_spy,
        "not_spy": p_feature_given_not_spy
    }

for feature, probs in spy_probs.items():
    print(feature)
    print(f"Probabilidades {probs}")
    print(probs['spy'] - probs['not_spy'])

travels_often
Probabilidades {'spy': np.float64(0.9), 'not_spy': np.float64(0.21428571428571427)}
0.6857142857142857
speaks_many_languages
Probabilidades {'spy': np.float64(0.7), 'not_spy': np.float64(0.35714285714285715)}
0.3428571428571428
uses_encryption
Probabilidades {'spy': np.float64(0.8), 'not_spy': np.float64(0.14285714285714285)}
0.6571428571428573
combat_training
Probabilidades {'spy': np.float64(0.7), 'not_spy': np.float64(0.21428571428571427)}
0.48571428571428565
suspicious_contacts
Probabilidades {'spy': np.float64(0.8), 'not_spy': np.float64(0.2857142857142857)}
0.5142857142857143
fake_identity
Probabilidades {'spy': np.float64(0.8), 'not_spy': np.float64(0.07142857142857142)}
0.7285714285714286
carries_weapon
Probabilidades {'spy': np.float64(0.7), 'not_spy': np.float64(0.14285714285714285)}
0.5571428571428572
avoids_social_media
Probabilidades {'spy': np.float64(0.7), 'not_spy': np.float64(0.21428571428571427)}
0.48571428571428565
works_in_tech
Probabilidades {'spy': n

In [146]:
# Function to predict spies
def predict_spy(person : dict, priors : dict, feature_probs : dict):
    spy_contributions = {}
    not_spy_contributions = {}
    score_spy = priors["spy"]
    score_not_spy = priors["not_spy"]
    
    for feature, value in person.items():
        p_spy = feature_probs[feature]['spy']
        p_not_spy = feature_probs[feature]['not_spy']

        if value == 1:
            score_spy *= p_spy
            score_not_spy *= p_not_spy
            spy_contributions[feature] = p_spy 
            not_spy_contributions[feature] = p_not_spy
        else:
            spy_contributions[feature] = (1 - p_spy)
            not_spy_contributions[feature] = (1 - p_not_spy)
            score_spy *= (1 - p_spy)
            score_not_spy *= (1 - p_not_spy)

    ratio = score_spy / score_not_spy
    max_score_spy_contributions = max(spy_contributions, key=spy_contributions.get)
    max_score_not_spy_contributions = max(not_spy_contributions, key=not_spy_contributions.get)

    if score_not_spy < score_spy:

        prediction = f'The person is spy, with a ratio of {ratio} and the feature that contributed the most is {max(spy_contributions, key=spy_contributions.get)} \n the spy score was {score_spy}  and the not spy score was {score_not_spy}'
    else:
        prediction = f'The person is not spy, with a ratio of {ratio} and the feature that contributed the most is {max(not_spy_contributions, key=not_spy_contributions.get)} \n the spy score was {score_spy}  and the not spy score was {score_not_spy}'
    
    return prediction, max_score_spy_contributions, max_score_not_spy_contributions


In [147]:
# New persons
persons = [
    {
"travels_often":1,
"speaks_many_languages":1,
"uses_encryption":1,
"combat_training":1,
"suspicious_contacts":1,
"fake_identity":1,
"carries_weapon":1,
"avoids_social_media":1,
"works_in_tech":0,
"spy_in_family":1
},
{
"travels_often":0,
"speaks_many_languages":0,
"uses_encryption":0,
"combat_training":0,
"suspicious_contacts":0,
"fake_identity":0,
"carries_weapon":0,
"avoids_social_media":0,
"works_in_tech":1,
"spy_in_family":0
},
{
"travels_often":1,
"speaks_many_languages":1,
"uses_encryption":1,
"combat_training":0,
"suspicious_contacts":0,
"fake_identity":0,
"carries_weapon":0,
"avoids_social_media":1,
"works_in_tech":1,
"spy_in_family":1
}
]

In [148]:
# Test
priors = {
    'spy' : prior_spy_1,
    'not_spy': prior_spy_0
}
for person in persons:
    prediction,max_spy_score, max_not_spy_score = predict_spy(person, priors, spy_probs)
    print(prediction)

The person is spy, with a ratio of 296320.18201488064 and the feature that contributed the most is travels_often 
 the spy score was 0.02124251136  and the not spy score was 7.168769678648903e-08
The person is not spy, with a ratio of 4.457872892577392e-06 and the feature that contributed the most is fake_identity 
 the spy score was 2.073599999999999e-07  and the not spy score was 0.04651545815612327
The person is spy, with a ratio of 1.2686749622443858 and the feature that contributed the most is travels_often 
 the spy score was 0.00016257023999999997  and the not spy score was 0.00012814175800584922


**Reflection**

Answer the following questions:

* What do the prior probabilities reveal about your dataset?
* Which features were **most informative** in your dataset?
* Did you observe any **zero-probability issues** before smoothing?
* Did Laplace smoothing affect any probability estimates?
* Do you think the **independence assumption** holds for the features you designed? Can you identify two features that might be correlated?
* Regarding the ambiguous test examples, what prediction does the model make?

My prior probabilities are 0.4 for spy and 0.6 for not spy, this reveal that i have more information about the people that aren't spies than who really are spies.

The most important feature (Who has the biggest impact) is the **"fake_identity"** feature with the biggest difference between $P(X_j=1 \mid \text{spy}=1)$ and $P(X_j=1 \mid \text{spy}=0)$.

Yes, we observed that the feature **"fake_identity"** had problems zero-probability issues before smoothing.

Yes, ignoring the features with zero probability issues, the rest of the probability estimates change when we did the smoothing, in a small way.

The independence assumption doesn't hold by itself for the features that we designed. In my opinion the 2 features that are the most correlated are **"carries_weapon"** and **"combat_training"**, but **"travels_often"** and **"speaks_many_languages"** could also be correlated too.

When we applied the prediction model to the ambiguous test example, the classifier predicted that the individual was a spy.


